# Make Model Figure

Read VTK file and create model figure

**Author:** Elco Luijendijk

In [ ]:
import matplotlib
matplotlib.use('Agg')

import matplotlib.patches
import matplotlib.collections

import sys
import os
import itertools
import random

import numpy as np
import matplotlib.pyplot as pl
import scipy.interpolate

import lib.read_vtu_file

from lib.grompy_lib import get_normal_flux_to_bnd

from model_input.figure_options import *

from matplotlib import ticker

## Figure Options

In [ ]:
# For direct file input, uncomment and set the path:
# vtk_file = '/path/to/your/file.vtu'
vtk_file = "model_output/median_case/vtk_files/runS0_L_8045.0_final_output.vtu"
vtkf_file = None

conc_only = False

## Helper Functions

In [ ]:
def add_subplot_axes(ax, rect, axisbg='w'):
    """
    embed a new subplot in existing subplot

    based on:
    http://stackoverflow.com/questions/17458580/embedding-small-plots-inside-subplots-in-matplotlib
    """

    fig = pl.gcf()
    box = ax.get_position()

    width = box.width
    height = box.height

    inax_position = ax.transAxes.transform(rect[0:2])
    transFigure = fig.transFigure.inverted()
    infig_position = transFigure.transform(inax_position)

    x = infig_position[0]
    y = infig_position[1]

    width *= rect[2]
    height *= rect[3]

    subax = fig.add_axes([x, y, width, height])

    x_labelsize = subax.get_xticklabels()[0].get_size()
    y_labelsize = subax.get_yticklabels()[0].get_size()

    x_labelsize *= rect[2] ** 0.5
    y_labelsize *= rect[3] ** 0.5

    subax.xaxis.set_tick_params(labelsize=x_labelsize)
    subax.yaxis.set_tick_params(labelsize=y_labelsize)

    return subax

In [ ]:
def project_flux_to_raster(element_x, element_y, xy_pts, bnd_ind,
                           arrow_x_int=250.0, arrow_y_int=10.0):

    """
    interpolate fluid flux to a regular raster

    """

    # convert fluxes to regular grid to generate equally spaced flow arrows
    topxy = xy_pts[:, :][bnd_ind]

    # flow

    topx_grid = np.arange(topxy[:, 0].min(), topxy[:, 0].max() + arrow_x_int,
                           arrow_x_int)
    node_order = np.argsort(topxy[:, 0])
    topy_grid = np.interp(topx_grid,
                           topxy[:, 0][node_order],
                           topxy[:, 1][node_order])

    # find thickness
    xmin_ind = xy_pts[:, 0] == xy_pts[:, 0].min()
    ys_xmin = xy_pts[:, 1][xmin_ind]
    thickness = ys_xmin.max() - ys_xmin.min()

    # find surface for each element loc
    element_surface = np.interp(element_x,
                                topxy[:, 0][node_order],
                                topxy[:, 1][node_order])
    element_depth = element_y - element_surface

    # interpolate qx and qy to regular depth grid
    qxi = topx_grid

    qyi = np.arange(0, -thickness+arrow_y_int, -arrow_y_int)

    # interpolate pressure on regular grid
    xgq, ygq = np.meshgrid(qxi, qyi)
    xgqf, ygqf = xgq.flatten(), ygq.flatten()

    # interpolate u to grid
    xyq_pts = np.vstack((element_x, element_depth)).T
    qx_interp = scipy.interpolate.griddata(xyq_pts,
                                           qx,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')
    qy_interp = scipy.interpolate.griddata(xyq_pts,
                                           qy,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')

    # project back to surface
    ygqf_surface = np.interp(xgqf,
                             topxy[:, 0][node_order],
                             topxy[:, 1][node_order])
    ygqf_proj = ygqf + ygqf_surface

    return xgqf, ygqf_proj, qx_interp, qy_interp

In [ ]:
def convert_to_grid(x, y, qx, qy, dx=1.0, dy=1.0):

    """
    interpolate a variable to a regular raster

    """

    # interpolate pressure on regular grid
    xi = np.arange(x.min(), x.max() + dx, dx)
    yi = np.arange(y.min(), y.max() + dy, dy)
    xg, yg = np.meshgrid(xi, yi)
    xgqf, ygqf = xg.flatten(), yg.flatten()

    # interpolate u to grid
    xyq_pts = np.vstack((x, y)).T
    qx_interp = scipy.interpolate.griddata(xyq_pts,
                                           qx,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')
    qy_interp = scipy.interpolate.griddata(xyq_pts,
                                           qy,
                                           np.vstack((xgqf, ygqf)).T,
                                           method='linear')

    qx_interp1 = np.resize(qx_interp, xg.shape)
    qy_interp1 = np.resize(qy_interp, xg.shape)

    return xg, yg, qx_interp1, qy_interp1

## File Setup and Configuration

In [ ]:
# Handle VTK file setup
if vtk_file is not None and 'vtu' in vtk_file:
    try:
        assert vtk_file[-4:] == '.vtu'
    except AssertionError:
        raise NameError('file name does not end with .vtu, are you sure this is a grompy/escript VTK file?')

    if '_Elements.vtu' not in vtk_file:
        vtk_file = vtk_file.split('.vtu')[0] + '_Elements.vtu'

    if vtkf_file is None:
        vtkf_file = vtk_file.split('_Elements.vtu')[0] + '_FaceElements.vtu'

    vtk_files = [vtk_file]
    vtkf_files = [vtkf_file]
    folder = os.path.split(vtk_file)[0]
else:
    raise ValueError('Please specify a valid VTK file')

print(f'Using VTK file: {vtk_file}')
print(f'Using VTK face file: {vtkf_file}')

In [ ]:
# Create output directory
folder_base = os.path.split(folder)[0]

fig_folder = os.path.join(folder_base, 'fig')
path = os.path.normpath(folder_base)
b = path.split(os.sep)

if os.path.exists(fig_folder) is False:
    os.mkdir(fig_folder)

print(f'Output directory: {fig_folder}')

## Main Processing

In [ ]:
# Process each VTK file
for vtk_file_path, vtkf_file_path in zip(vtk_files[::-1], vtkf_files[::-1]):

    fn = vtk_file_path

    print(f'Reading VTK file: {fn}')

    xy_pts, conn, pt_var_names, pt_var_arrays, cell_var_names, cell_var_arrays = \
        lib.read_vtu_file.read_vtu_file(fn)

    fnf = vtkf_file_path

    print(f'Reading VTK face file: {fnf}')

    xy_pts_face, conn_face, pt_var_names_face, pt_var_arrays_face, cell_var_names_face, cell_var_arrays_face = \
        lib.read_vtu_file.read_vtu_file(fnf)

    vtk_filename = os.path.split(vtk_file_path)[-1]
    vtkf_filename = os.path.split(vtkf_file_path)[-1]

    # construct label from filename
    f = vtk_filename.split('.')[:-1]
    f = ''.join(f)
    f1 = f.split('_')
    ind = f1.index('Elements')
    f2 = f1[:ind]
    if f2[-1] == 'output':
        f2 = f2[:-1]

    f3 = f2[0].split('S')

    plot_title = None  # Initialize plot_title
    if add_title is True:
        plot_title = ' '.join(f3)
        plot_title += ', '
        plot_title += ' '.join(f2[1:-2])
        plot_title += '= %s' % f2[-2]
        if f2[-1] == 'final':
            plot_title += ', final timestep'
        else:
            f4 = f2[-1].split('s')
            plot_title += ', timestep %s' % f4[-1]

    # get fluxes
    qx_ind = cell_var_names.index('qx')
    qx = cell_var_arrays[qx_ind]
    qy_ind = cell_var_names.index('qy')
    qy = cell_var_arrays[qy_ind]

    # make pts 2d
    xy_pts = xy_pts[:, :2]
    xy_pts_face = xy_pts_face[:, :2]

    # make closed form of polygons
    conn_cl = np.zeros((conn.shape[0], conn.shape[1] + 1), dtype=int)
    conn_cl[:, :conn.shape[1]] = conn
    conn_cl[:, -1] = conn[:, 0]

    print('Finished reading files')

In [ ]:
# Extract boundary fluxes
print('Finding top boundary fluxes')

# Find flux indices
f_ind = None
if 'nodal_flux_surface' in pt_var_names:
    f_ind = pt_var_names.index('nodal_flux')
elif 'nodal_flux' in pt_var_names:
    f_ind = pt_var_names.index('nodal_flux')

if f_ind is None:
    raise ValueError('Could not find nodal_flux or nodal_flux_surface in point variables')

nfx = pt_var_arrays[f_ind][:, 0]
nfy = pt_var_arrays[f_ind][:, 1]
nodal_flux = pt_var_arrays[f_ind].copy()

# Scale by year
nodal_flux *= year

nodata = -99999
if 'surface' in pt_var_names:
    sf_ind = pt_var_names.index('surface')
    bnd_ind = pt_var_arrays[sf_ind] == 1
else:
    sf_ind = pt_var_names.index('nodal_flux_surface')
    nfx = pt_var_arrays[sf_ind][:, 0]
    nfy = pt_var_arrays[sf_ind][:, 1]
    bnd_ind = (nfx > nodata) & (nfy > nodata)

topxy = xy_pts[:, :][bnd_ind]
x_order = np.argsort(topxy[:, 0])
top_xy_sorted = topxy[x_order]
nodal_flux_top_sorted = nodal_flux[bnd_ind][x_order]

slope = np.diff(top_xy_sorted[:, 1]) / np.diff(top_xy_sorted[:, 0])
slope_mean = slope.mean()

# find h at top bnd
h = pt_var_arrays[pt_var_names.index('h')]
h_top = h[bnd_ind][x_order]

print('Rotating flux')
nodal_flux_rotated = \
    get_normal_flux_to_bnd(nodal_flux_top_sorted,
                           top_xy_sorted)

# correct for surpressed recharge at seepage bnd nodes
est_rch_flux = nodal_flux_rotated[:, 1][-1]

ind = (top_xy_sorted[:, 0] > 0) & (nodal_flux_rotated[:, 1] > (0.01 * est_rch_flux))
nodal_flux_corrected = nodal_flux_rotated.copy()

# convert flux to m/yr
nfx *= year
nfy *= year

print('Boundary fluxes extracted')

In [ ]:
# Calculate element centers and create polygons
print('Generating polygons for elements')
polys = [xy_pts[conn_i] for conn_i in conn]
patches = [matplotlib.patches.Polygon(poly) for poly in polys]

print('Generating polygons for face elements')
polys_face = [xy_pts_face[conn_i] for conn_i in conn_face]

# calculate polygon centre
polys_array = np.array(polys)
polys_array_face = np.array(polys_face)

element_x = np.mean(polys_array[:, :, 0], axis=1)
element_y = np.mean(polys_array[:, :, 1], axis=1)

element_x_face = np.mean(polys_array_face[:, :, 0], axis=1)
element_y_face = np.mean(polys_array_face[:, :, 1], axis=1)

print('Element centers calculated')

In [ ]:
# Create figure
print('Creating figures')

fig, axs = pl.subplots(2, 1, gridspec_kw={'height_ratios': [1, 3], 'hspace':0.0}, 
                       sharex=True, figsize=(8,4), layout="constrained")
tax, ax = axs

for axi in axs:
    axi.spines['top'].set_visible(False)
    axi.spines['right'].set_visible(False)
    axi.get_xaxis().tick_bottom()
    axi.get_yaxis().tick_left()

cax2 = None
if add_colorbar is True:
    rect1 = [0.7, 0.05, 0.3, 0.25]
    cbbox = ax.inset_axes(rect1)
    [cbbox.spines[k].set_visible(False) for k in cbbox.spines]
    cbbox.tick_params(
        axis = 'both',
        left = False,
        top = False,
        right = False,
        bottom = False,
        labelleft = False,
        labeltop = False,
        labelright = False,
        labelbottom = False
    )

    cbbox.set_facecolor([1,1,1,0.7])
    cax2 = cbbox.inset_axes([0.1, 0.8, 0.8, 0.15])

print('Figure created')

In [ ]:
# Prepare visualization data
if conc_only is True:
    c_ind = None
    if 'concentration' in pt_var_names:
        c_ind = pt_var_names.index('concentration')
    elif 'conc' in pt_var_names:
        c_ind = pt_var_names.index('conc')

    if c_ind is None:
        print('Warning: concentration variable not found, plotting all variables')
        pt_var_arrays_plot = pt_var_arrays
        pt_var_names_plot = pt_var_names
    else:
        pt_var_arrays_plot = [pt_var_arrays[c_ind]]
        pt_var_arrays_plot[0] = np.round(pt_var_arrays_plot[0], decimals=3)
        pt_var_names_plot = [pt_var_names[c_ind]]
else:
    pt_var_arrays_plot = pt_var_arrays
    pt_var_names_plot = pt_var_names

print(f'Will plot {len(pt_var_names_plot)} variable(s)')

In [ ]:
# Plot variables - initialize tracking variables
elements = None
leg_sc = None
cb = None
leg_wt2 = None
bnd_flux = None
bnd_flux2 = None

for nplot, pt_var_name, pt_var_array in zip(itertools.count(),
                                            pt_var_names_plot,
                                            pt_var_arrays_plot):

    print(f'Plotting variable: {pt_var_name}')

    for vdim in range(len(pt_var_array.shape)):
        if len(pt_var_array.shape) > 1:
            # vector variable
            vtype = 'vector'
        else:
            vtype = 'scalar'

        if vtype == 'scalar':
            # calculate cell averages for model var
            cell_avg = np.array([pt_var_array[conn_i].mean()
                                 for conn_i in conn])
        else:
            cell_avg = np.array([pt_var_array[conn_i, vdim].mean()
                                 for conn_i in conn])

        if show_elements is True:
            # add element values:
            if nplot == 0:
                elements = matplotlib.collections.PatchCollection(
                    patches,
                    linewidths=0.0,
                    antialiased=False)
                ax.add_collection(elements)

            if elements is not None:
                elements.set_array(cell_avg)
                elements.set_clim(pt_var_array.min(), pt_var_array.max())
                elements.set_edgecolor('None')
                elements.set_zorder(0.1)
                elements.set_antialiased(True)
                elements.set_cmap(cmap)

        else:
            if vtype == 'scalar':
                if nplot == 0:
                    scatter_int = 1
                    leg_sc = ax.scatter(xy_pts[::scatter_int, 0],
                                        xy_pts[::scatter_int, 1],
                                        c=pt_var_array[::scatter_int],
                                        edgecolor='None', s=7)

                else:
                    if leg_sc is not None:
                        leg_sc.set_array(pt_var_array[::scatter_int])
                        if add_colorbar is True and cb is not None:
                            cb.set_clim(pt_var_array.min(),
                                        pt_var_array.max())

        xlim = [xy_pts[:, 0].min(), xy_pts[:, 0].max()]
        if ylim_fixed is None:
            ylim = [xy_pts[:, 1].min(), xy_pts[:, 1].max()]
        else:
            ylim = ylim_fixed

        if nplot == 0:
            ax.plot([xy_pts[:, 0].min(), 0], [0, 0],
                    color=sea_lvl_color, lw=0.25)

        if nplot == 0:
            va = (qx ** 2 + qy ** 2) ** 0.5
            scale = np.abs(va).max() * 10.0

            print('Converting flux to regular grid')
            xg, yg, qxg, qyg = convert_to_grid(element_x, element_y,
                                               qx * year,
                                               qy * year, dx=dx, dy=dy)

            mask = np.zeros_like(qxg, dtype=bool)
            mask[np.isnan(qxg)] = True

            qxgm = np.ma.array(qxg, mask=mask)
            qygm = np.ma.array(qyg, mask=mask)
            vag = (qxgm ** 2 + qygm ** 2) ** 0.5
            lw = lw_min + lw_max * vag / vag.max()

            leg_splot = ax.streamplot(xg, yg, qxgm, qygm,
                                      density=streamline_density,
                                      color='k', linewidth=lw)

            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.set_xlabel('Distance (m)')
            ax.set_ylabel('Elevation (m)')

In [ ]:
        # Continue plotting boundary flux
        if nplot == 0:
            top_x = xy_pts[:, 0][bnd_ind]
            top_flux = nfy[bnd_ind]
            x_order_top = np.argsort(top_x)
            y0 = np.zeros_like(top_xy_sorted[:, 0])

            bnd_flux = \
                tax.fill_between(top_xy_sorted[:, 0],
                                 y0,
                                 nodal_flux_corrected[:, 1],
                                 where=nodal_flux_corrected[:, 1] > 0,
                                 facecolor='lightblue',
                                 edgecolor='black',
                                 lw=0.5)
            bnd_flux2 = \
                tax.fill_between(top_x[x_order_top],
                                 y0,
                                 nodal_flux_corrected[:, 1],
                                 where=nodal_flux_corrected[:, 1] < 0,
                                 facecolor='blue',
                                 edgecolor='black',
                                 lw=0.5)

            if debug is True:
                if 'flux_surface_norm' in cell_var_names_face:
                    ind = cell_var_names_face.index('flux_surface_norm')
                    ind_sort = np.argsort(element_x_face)
                    bnd_flux3, = tax.plot(element_x_face[ind_sort],
                                          cell_var_arrays_face[ind][:, 1][ind_sort],
                                          color='green', lw=0.5, zorder=1001)

                elif 'flux_surface_norm' in pt_var_names:
                    ind = pt_var_names.index('flux_surface_norm')
                    bnd_flux3 = tax.scatter(xy_pts[:, 0],
                                            pt_var_arrays[ind][:, 1],
                                            color='green', s=1, zorder=1001)

            if debug is True:
                print('Showing location of seepage boundary')
                if 'active_seepage_bnd' in pt_var_names:
                    ind = pt_var_names.index('active_seepage_bnd')
                    ind2 = pt_var_arrays[ind] > 0
                    xys = xy_pts[ind2]
                    zs = np.zeros((xys.shape[0]))
                    tax.scatter(xys[:, 0], zs, color='brown', s=2, zorder=1003)

            tax.set_xlim(xlim)
            tax.axhline(y=0, color='black', lw=0.5)
            tax.axvline(x=0, color='black', lw=0.5)
            tax.set_ylabel('Boundary flux (m/yr)')

            if add_title is True and plot_title is not None:
                tax.set_title(plot_title, fontsize='small')

            if tax_ylim_fixed is not None:
                tax.set_ylim(tax_ylim_fixed)
            else:
                tax.set_ylim(nodal_flux_corrected[:, 1].min() * 1.25,
                             nodal_flux_corrected[:, 1].max() * 1.25)

            if add_colorbar is True and cax2 is not None:
                if show_elements is True:
                    cb = fig.colorbar(elements,
                                      cax=cax2,
                                      orientation='horizontal')
                else:
                    if leg_sc is not None:
                        cb = fig.colorbar(leg_sc,
                                          cax=cax2,
                                          orientation='horizontal')
                if cb is not None:
                    cb.ax.tick_params(labelsize=leg_fontsize)

                    # (generate plot here)
                    tick_locator = ticker.MaxNLocator(nbins=3)
                    cb.locator = tick_locator
                    cb.update_ticks()

In [ ]:
        # Set colorbar labels
        if add_colorbar is True and cb is not None:
            if 'concentration' in pt_var_name:
                cb.set_label('Solute concentration (kg/kg)', fontsize='small')
            else:
                cb.set_label(pt_var_name)

        if nplot == 0:
            leg_wt2, = ax.plot(top_xy_sorted[:, 0], h_top,
                               color='gray', lw=1.5)

            if debug is True:
                print('Hydraulic head at top nodes: ', h_top)
            # add vertical line at coastline
            ax.axvline(x=0, color='black', lw=0.5, zorder=0)

        if nplot == 0 and add_legend is True:
            if bnd_flux2 is not None and bnd_flux is not None and leg_wt2 is not None:
                legs = [bnd_flux2, bnd_flux, leg_wt2]
                labels = ['recharge', 'discharge', 'watertable']
                tax.legend(legs, labels, frameon=False, loc='upper right', fontsize=leg_fontsize, ncol=3)

        # save regular figure:
        file_basename = ''.join(vtk_filename.split('.')[:-1])
        if vtype == 'scalar':
            fig_fn = file_basename + '_' + pt_var_name + figure_type
        else:
            fig_fn = file_basename + '_' + pt_var_name \
                     + ['x', 'y', 'z'][vdim] + figure_type

        fn_out = os.path.join(fig_folder, fig_fn)
        print(f'Saving figure: {fn_out}')
        fig.savefig(fn_out, dpi=dpi)

print('Done processing file')

pl.clf()
print('All done!')